In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

In [ ]:
data_path = '/kaggle/input/arcade/arcade_challenge_datasets/dataset_phase_1/segmentation_dataset'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
import os
import numpy as np
import albumentations as A

train_transform = A.Compose([
    A.Resize(width=512, height=512),
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=1),
    A.ColorJitter(brightness=0.5, contrast=1, saturation=0.1, hue=0.5)
])

valid_transform = A.Compose([
    A.Resize(width=512, height=512)
])

def create_mask(img_id, coco):
    anns_ids = coco.getAnnIds(imgIds=img_id, iscrowd=None)
    anns = coco.loadAnns(anns_ids)
    mask = coco.annToMask(anns[0])
    
    for i in range(len(anns)):
        mask+=coco.annToMask(anns[i])
    
    mask = (mask>=1).astype(np.float32)
    
    return mask
    
def load_dataset(dir_path, coco, it = 5):
    img = []
    mask = []
    
    imgs_dict = coco.imgs
    
    for img_item in imgs_dict.values():
        img.append(os.path.join(dir_path,img_item['file_name']))
        mask.append(img_item['id'])
    
    return img, mask

In [ ]:
from PIL import Image
from pycocotools.coco import COCO
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2

class VCDDataset(Dataset):
    def __init__(self, dir_path, coco, transform=None):
        X,y = load_dataset(dir_path, coco)
        self.X = X
        self.y = y
        self.coco = coco
        self.transform = transform
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self,i):
        img = Image.open(self.X[i]).convert('L')
        img = np.array(img)

        mask = create_mask(self.y[i], self.coco)
        old_mask = mask

        if self.transform:
            sample = self.transform(image=img, mask=mask)
            img = sample['image']
            mask = sample['mask']
            mask = v2.ToTensor()(mask)
            img = v2.ToTensor()(img)
            
        return img, mask, old_mask

train_data_path = os.path.join(data_path,'seg_train/images')
valid_data_path = os.path.join(data_path,'seg_val/images')

coco_train = COCO(os.path.join(data_path,'seg_train/annotations/seg_train.json'))
coco_valid = COCO(os.path.join(data_path,'seg_val/annotations/seg_val.json'))

train_dataset = VCDDataset(train_data_path, coco_train, transform=train_transform)
valid_dataset = VCDDataset(valid_data_path, coco_valid, transform = valid_transform)  

len(train_dataset), len(valid_dataset)

In [ ]:
train_dataloader = DataLoader(train_dataset,batch_size=8, shuffle=True)
valid_dataloader = DataLoader(valid_dataset,batch_size=8)

In [ ]:
import matplotlib.pyplot as plt
import torchvision.transforms as t

for item in train_dataloader:
    img, mask, old_mask = item
    print(img.type())
    print(mask.unique())
    for i in range(len(img)):
        fig, ax = plt.subplots(1, 3)
        
        orig_img = t.ToPILImage()(img[i])
        orig_mask  = t.ToPILImage()(mask[i])
        orig_old_mask = t.ToPILImage()(old_mask[i])
        
        ax[0].imshow(orig_img, cmap='gray')
        ax[0].set_title(f'img {i}')

        ax[1].imshow(orig_mask, cmap='gray')
        ax[1].set_title(f'mask {i}')

        ax[2].imshow(orig_old_mask, cmap='gray')
        ax[2].set_title(f'old mask {i}')
        plt.show()
    break
        
    

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self,in_c,out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c,out_c,(3,3),padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c,out_c,(3,3),padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
            )
    
    def forward(self,x):
        return self.conv(x)

class DownConv(nn.Module):
    def __init__(self,in_c, out_c):
        super().__init__()
        self.conv = DoubleConv(in_c,out_c)
        self.pool =  nn.MaxPool2d((2,2),stride=2)
    
    def forward(self,x):
        skip_conn = self.conv(x)
        return skip_conn, self.pool(skip_conn)

class UpConv(nn.Module):
    def __init__(self,in_c,out_c):
        super().__init__()
        self.in_c = in_c
        self.out_c = out_c
        self.up = nn.ConvTranspose2d(in_c,out_c,(2,2),stride=2)
        self.conv = DoubleConv(in_c,out_c)
    
    def forward(self,x1,x2):
        x2 = self.up(x2)
        x = torch.cat([x1,x2],dim=1)
        return self.conv(x)

In [ ]:
class Unet(nn.Module):
    def __init__(self):
        super().__init__()

        self.down1 = DownConv(1,64)
        self.down2 = DownConv(64,128)
        self.down3 = DownConv(128,256)
        self.down4 = DownConv(256,512)

        self.bottle = DoubleConv(512,1024)

        self.up1 = UpConv(1024,512)
        self.up2 = UpConv(512,256)
        self.up3 = UpConv(256,128)
        self.up4 = UpConv(128,64)

        self.final = nn.Conv2d(64,1,(1,1))

    def forward(self,x):

        s1,x = self.down1(x)
        s2,x = self.down2(x)
        s3,x = self.down3(x)
        s4,x = self.down4(x)

        x = self.bottle(x)

        x = self.up1(s4,x)
        x = self.up2(s3,x)
        x = self.up3(s2,x)
        x = self.up4(s1,x)

        return nn.Sigmoid()(self.final(x))


In [ ]:
from tqdm import tqdm

model = Unet()
model.to(device)
optim = torch.optim.AdamW(model.parameters(),lr=5e-4)
criterion = nn.BCELoss().to(device)
epochs = 50
loss_values = []
vloss_values = []
min_loss = 10000

for epoch in tqdm(range(epochs)):
    print(f'epochs:{epoch}')
    running_loss = 0.0
    model.train(True)
    for i, item in enumerate(train_dataloader):
        img, mask, _ = item
        mask = mask.to(device)
        img = img.to(device)
        out = model(img)
        
        loss = criterion(out,mask)
        
        loss.backward()
        optim.step()
        optim.zero_grad()
        
        running_loss += loss.item()
        loss_values.append(loss.item())
    
    model.eval()
    running_vloss = 0.0
    with torch.no_grad():
        for i, item in enumerate(valid_dataloader):
            img, mask, _ = item
            img = img.to(device)
            mask = mask.to(device)
            
            out = model(img)
            
            loss = criterion(out,mask)
            running_vloss += loss.item()  
            vloss_values.append(loss.item())
    print(f'train and valid loss after epoch: {loss_values[-1]:.5f}, {vloss_values[-1]:.5f}')
    
    if min_loss > vloss_values[-1]:
        min_loss = vloss_values[-1]
        torch.save(model.state_dict(), 'model.pth')
        print('model saved')

In [ ]:
import matplotlib.pyplot as plt

plt.plot(loss_values,label='loss')
plt.plot(vloss_values,label = 'valid')
plt.legend()
plt.show()

In [ ]:
model = Unet()
model.load_state_dict(torch.load('/kaggle/input/unet-arcade-bce/pytorch/default/1/model.pth'))
model.to(device)
model.eval()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def postporcessing(tensor):
    min_val = tensor.min()
    max_val = tensor.max()
    norm_tensor = (tensor - min_val) / (max_val - min_val)
    bin_tensor = norm_tensor.float()
    return bin_tensor
    
fig, ax = plt.subplots(1, 3, figsize=(15,5))
it = 0

for i,data in enumerate(valid_dataloader):
    if it < i:
        continue
    img, mask, _ = data
    for item in img:
        ax[0].imshow(item.permute(1,2,0), cmap='gray')
        ax[0].axis('off')
        ax[0].set_title('image')
        
        item = item.unsqueeze(0)
        item = item.to(device)
        out = model(item)
        img = out[0].cpu().detach().permute(1,2,0)
        img_norm = postporcessing(img)
        
        ax[1].imshow(img_norm,cmap='gray')
        ax[1].set_title('unet')
        ax[1].axis('off')

        img_norm = (img_norm.numpy()*255).astype(np.uint8)
        canny = cv2.Canny(img_norm,127,255)
        ax[2].imshow(canny, cmap='gray')
        ax[2].axis('off')
        ax[2].set_title('unet + canny')
        break
        
    for item in mask:
        item = item.to(device)
        img = t.ToPILImage()(item)
        img = item.cpu().detach().permute(1,2,0)
        print(img.shape)
        ax[3].imshow(img,cmap='gray')
        ax[3].set_title('mask')
        break
    plt.show
    plt.savefig(f'out/{it}.png', format='png', transparent=False, bbox_inches='tight')
    if it == i:
        it+=1
        break